--Вывести аэропорты, из которых выполняется менее 50 рейсов



In [ ]:
select 
	f.departure_airport,
	a.airport_name, 
	count(*) as cnt
from flights f 
join airports a  on a.airport_code = f.departure_airport  
group by f.departure_airport, a.airport_name
having count(*) < 50

--Вывести среднюю стоимость билетов для каждого маршрута (город вылета - город прилета)



In [ ]:
select 
	a_dep.city ,
	a_ar.city   ,
	tf.fare_conditions,
	round(avg(tf.amount ), 0)
from ticket_flights tf 
join flights f on f.flight_id = tf.flight_id 
join airports a_dep on a_dep.airport_code  = f.departure_airport
join airports a_ar on a_ar.airport_code = f.arrival_airport 
group by a_dep.airport_code ,
	a_ar.airport_code , tf.fare_conditions 
order by 1,2

-- Вывести топ-5 самых загруженных маршрутов (по количеству проданных билетов)




In [ ]:
select  
	f.departure_airport,
	f.arrival_airport ,
	tf.flight_id,
	count(*)
from ticket_flights tf 
join flights f on f.flight_id = tf.flight_id 
group by tf.flight_id, f.departure_airport,
	f.arrival_airport 
order by count(*) desc
limit 5

-- Найти пары рейсов, вылетающих из одного аэропорта в течение 1 часа
-- Подсказка: (f1.scheduled_departure - f2.scheduled_departure))) <= 3600




In [ ]:
select 
	f1.departure_airport,
	extract (epoch from (f2.scheduled_departure - f1.scheduled_departure)) /60
from flights f1 
join flights f2 on f2.departure_airport  = f1.departure_airport  
where f1.flight_id < f2.flight_id
	and f2.scheduled_departure between f1.scheduled_departure 
		and f1.scheduled_departure + interval '1 hour';
 

'Вам нужно проанализировать данные о продажах билетов, чтобы получить статистику в следующих разрезах:
- По классам обслуживания (fare_conditions)
- По месяцам вылета
- По аэропортам вылета
- По комбинациям: класс + месяц, класс + аэропорт, месяц + аэропорт
- Общие итоги'



In [ ]:
select 
	tf.fare_conditions,
	f.departure_airport,
	extract (month from f.scheduled_departure) as month_dep, 
	count(*)
from ticket_flights tf 
join flights f on f.flight_id = tf.flight_id 
group by grouping sets (
				(tf.fare_conditions),
				(month_dep),
				(f.departure_airport),
				(tf.fare_conditions, month_dep),
				(tf.fare_conditions, f.departure_airport),
				(month_dep, f.departure_airport)
)


'Рейсы с задержкой больше средней (через CTE).
Найдите рейсы, задержка которых превышает среднюю задержку по всем рейсам.'

    чтобы не мудрить в типами, решил использовать подзапрос






In [ ]:
with avg_flig_del as (select avg(f.actual_arrival - f.scheduled_arrival  ) from flights f ) 


select 
	*,
	f.actual_arrival - f.scheduled_arrival
from flights f 
--cross join avg_flig_del 
where  (f.actual_arrival - f.scheduled_arrival ) > (select avg(f.actual_arrival - f.scheduled_arrival  ) from flights f ) 

